# MatPES-r2SCAN FP Force Evaluation and SLURM Generator

This notebook is a **reusable FP force-evaluation and SLURM-generation workflow**, not a one-off script. The values currently set in it reproduce this study's configuration for the MatPES-r2SCAN dataset on the Zaratan (UMD) cluster.

**Everything dataset/cluster-specific is a variable, not a hardcoded assumption:** dataset paths, chunk count, filename prefix, model selection, checkpoint paths, output paths, virtual environments, and SLURM resources are all set in Section 3 (and the registry in Section 2). To reuse this workflow for another dataset or on another machine, update those parameters — the chunk file layout, provenance fields (`structure_id`/`atom_index`), and the standardized-output contract (Section 7) stay the same either way.

**Why one registry entry per model, with its own Python/venv/site-packages:** the r2SCAN workflow evaluates only the three r2SCAN-trained models (M3GNet-MatPES-r2SCAN, TensorNet-MatPES-r2SCAN, MACE-MatPES-r2SCAN) — never the PBE-trained models used for the other two datasets. Their PyTorch/ASE/package-version requirements can conflict with each other, so we recommend a separate tested virtual environment per FP family. Every `POTENTIAL_REGISTRY` entry therefore records its own `python` executable, `venv_activate` script, `site_pkgs` directory, checkpoint (`model_path`), and `calc_setup_code` — nothing is assumed to be shared between models.

**This notebook never installs packages automatically** — if a required package is missing in a given environment, cells raise a clear error instead of running `pip install` for you.

**Zero-force exclusion (Methods):** atoms where the DFT or FP force is ~zero have an undefined force *angle*, so they are excluded from every flattened per-atom metric array (`force_angle_error` and friends) — this is the finite-`force_angle_error` mask from the manuscript, applied independently per potential (a zero FP force is model-specific, so different potentials can retain slightly different atom populations; this is expected, not an inconsistency). The raw per-chunk checkpoint (`results_*.json`) still retains **every** atom, with the undefined angle stored as JSON `null` — nothing is ever silently dropped from the raw record, only from the flattened analysis arrays built on top of it.

| Section | Purpose |
|---|---|
| 0 — Dataset Chunker | Splits a raw dataset file into N chunks (skip if chunks already exist) |
| 1 — Imports & Helpers | Helper code (incl. the embedded force-error calculation) embedded into every generated script |
| 2 — Potential Registry | Shared registry across all three `matpes_*_run_generator` notebooks + this dataset's model subset |
| 3 — Configuration | Active potential, dataset/chunk/output paths, run mode, SLURM settings |
| 4 — Generate Scripts (one potential) | Writes run scripts + SLURM files + `mass_submit.sh` for `ACTIVE_POTENTIAL` |
| 5 — Validate & Merge (one potential) | Strict-by-default merge of that potential's chunk outputs |
| 6 — Generate + Submit (all potentials) | Same as Section 4/mass-submission, looped over this dataset's full model list |
| 7 — Strict All-Model Merge + Standardized Output | Cross-model consistency checks, then writes the one final `{STANDARDIZED_DATASET_NAME}_force_results_standardized.json` |

**Three output layers, not to be confused with each other:**
- **Per-chunk raw/intermediate**: `data_{fp}_{DATASET_LABEL}_chunk_XX.json` (flattened arrays) and `results_{fp}_{DATASET_LABEL}_chunk_XX.json` (full per-structure Cartesian checkpoint, every atom).
- **Per-model merged intermediate**: `all_results_{fp}_{DATASET_LABEL}.json` (Section 5/6).
- **One final standardized dataset file**: `{STANDARDIZED_DATASET_NAME}_force_results_standardized.json` (Section 7) — the only file the analysis notebooks load, via `standardized["models"]`, unmodified.

**Two run modes** (Section 3): `"chunked"` (one script per chunk, submitted in parallel) or `"single"` (one script loops over all chunks sequentially). The Section 6 mass-submission workflow currently supports `"chunked"` mode only — it walks `code_chunk_XX/` subdirectories.


---
## Section 0 — Dataset Chunker

Splits a raw MatPES r2SCAN dataset into N equally-sized chunks, writing the same directory layout used on the HPC cluster:

```
CHUNK_BASE_DIR/
  chunk00/  r2scan_00.json.gz
  chunk01/  r2scan_01.json.gz
  ...
  r2scan_manifest.json
```

**Chunks already exist on the HPC cluster** at:
`/path/to/dataset_chunks` (see `CHUNK_BASE_DIR` below)

**This cell cannot be run as-is.** `CHUNK_DATASET_PATH` below currently points at
that *existing chunk directory*, not a raw dataset file — the chunker only
accepts a single raw `.json`, `.json.gz`, or `.xyz` file (see the format checks
below), so running this cell against a directory will raise
`ValueError: Unsupported file format`. That is intentional: use the existing
chunks (Section 1 onward reads them directly) unless you specifically need to
re-chunk. If you do need to re-chunk (e.g. a new r2SCAN dataset drop), set
`CHUNK_DATASET_PATH` to the actual raw `MatPES-r2SCAN-*.json.gz` file first —
this notebook does not invent or guess that path for you.

**Skip to Section 1** unless you need to re-chunk with a different dataset or N.

| Parameter | Description |
|---|---|
| `CHUNK_DATASET_PATH` | Path to a raw `MatPES-r2SCAN-*.json.gz` / `.json` / `.xyz` file — **not** a chunk directory |
| `CHUNK_BASE_DIR` | Output directory for the chunks (created if absent) |
| `CHUNK_N` | Number of chunks (default: 20) |
| `CHUNK_FUNCTIONAL` | Fixed to `'r2scan'` — do not change |
| `CHUNK_N_MAX` | Cap total entries for a quick test; `None` = use all |

In [ ]:
from pathlib import Path

def _require_configured(value, name):
    s = str(value)
    if value is None or s.startswith("/path/to/") or s.startswith("your-"):
        raise ValueError(
            f"{name} has not been configured -- edit this cell and set {name} to "
            "the real value for your system before running this notebook."
        )
    return value

import gzip as _gzip
import os, json, hashlib  # also imported in the main imports cell below

# ── Configuration ─────────────────────────────────────────────────────────────
# NOTE: r2SCAN chunks already exist on HPC — skip this cell unless re-chunking.
CHUNK_DATASET_PATH = Path("/path/to/MatPES-r2SCAN-2025.1.json.gz")  # EDIT: raw dataset file (NOT the existing chunk dir -- see note above)
CHUNK_BASE_DIR     = Path("/path/to/dataset_chunks")                 # EDIT: output dir for chunks
_require_configured(CHUNK_DATASET_PATH, "CHUNK_DATASET_PATH")
_require_configured(CHUNK_BASE_DIR, "CHUNK_BASE_DIR")
CHUNK_DATASET_PATH = str(CHUNK_DATASET_PATH)
CHUNK_BASE_DIR     = str(CHUNK_BASE_DIR)
CHUNK_N            = 20
CHUNK_FUNCTIONAL   = "r2scan"   # fixed — do not change
CHUNK_N_MAX        = None
# ── Load entries ──────────────────────────────────────────────────────────────
print("Loading r2SCAN dataset (may take a minute)...")
_path_lower = CHUNK_DATASET_PATH.lower()
if _path_lower.endswith('.xyz'):
    from ase.io import read as _ase_read
    from pymatgen.io.ase import AseAtomsAdaptor as _ASEAdaptor
    _adaptor_s0 = _ASEAdaptor()
    _atoms_list = _ase_read(CHUNK_DATASET_PATH, index=':')
    all_entries = []
    for _atoms in _atoms_list:
        try:
            _forces = _atoms.get_forces().tolist()
            _energy = float(_atoms.get_potential_energy())
        except Exception:
            _forces = [[0.0, 0.0, 0.0]] * len(_atoms)
            _energy = 0.0
        _struct = _adaptor_s0.get_structure(_atoms)
        all_entries.append({
            'forces':         _forces,
            'energy':         _energy,
            'formula_pretty': _atoms.get_chemical_formula(),
            'structure':      _struct.as_dict(),
        })
elif _path_lower.endswith('.json.gz'):
    with _gzip.open(CHUNK_DATASET_PATH, 'rt') as _f:
        _raw = json.load(_f)
    all_entries = _raw.get('data', _raw) if isinstance(_raw, dict) else _raw
elif _path_lower.endswith('.json'):
    with open(CHUNK_DATASET_PATH, 'r') as _f:
        _raw = json.load(_f)
    all_entries = _raw.get('data', _raw) if isinstance(_raw, dict) else _raw
else:
    raise ValueError(f"Unsupported file format: {CHUNK_DATASET_PATH!r}  (must be .xyz, .json, or .json.gz)")

functional_lower = CHUNK_FUNCTIONAL.lower()
total_in_dataset = len(all_entries)

if CHUNK_N_MAX is not None:
    all_entries = all_entries[:CHUNK_N_MAX]
total_used = len(all_entries)
all_indices = list(range(total_used))

print(f"Label (file prefix) : '{functional_lower}'")
print(f"Total in dataset    : {total_in_dataset:,}")
print(f"Total to chunk      : {total_used:,}  \u2192  ~{total_used // CHUNK_N:,} per chunk")

# ── Write chunks ──────────────────────────────────────────────────────────────
# Each entry additionally carries 'structure_id' (== 'original_index') so
# downstream loaders can standardize on one name; 'original_index' is kept
# for backward compatibility with chunks already generated (purely additive --
# existing chunks are never touched or regenerated by this cell).
os.makedirs(CHUNK_BASE_DIR, exist_ok=True)
base_size = total_used // CHUNK_N
remainder = total_used  % CHUNK_N
start = 0
chunk_sizes           = []   # index = chunk_id
per_chunk_id_checksum = {}   # {chunk_id: sha256 of that chunk's sorted structure_ids}

def _ids_checksum(ids):
    return hashlib.sha256(",".join(str(x) for x in sorted(ids)).encode("utf-8")).hexdigest()

for chunk_id in range(CHUNK_N):
    this_size     = base_size + (1 if chunk_id < remainder else 0)
    end           = start + this_size
    chunk_idxs    = all_indices[start:end]
    data_with_idx = [dict(e, original_index=i, structure_id=i) for i, e in zip(chunk_idxs, all_entries[start:end])]
    payload = {
        'functional':       CHUNK_FUNCTIONAL,
        'total_in_dataset': total_in_dataset,
        'total_used':       total_used,
        'chunk_id':         chunk_id,
        'n_chunks':         CHUNK_N,
        'chunk_size':       this_size,
        'original_indices': chunk_idxs,   # kept for backward compatibility
        'structure_ids':    chunk_idxs,
        'data':             data_with_idx,
    }
    chunk_dir = os.path.join(CHUNK_BASE_DIR, f'chunk{chunk_id:02d}')
    os.makedirs(chunk_dir, exist_ok=True)
    out_file  = os.path.join(chunk_dir, f'{functional_lower}_{chunk_id:02d}.json.gz')
    with _gzip.open(out_file, 'wt', encoding='utf-8') as _f:
        json.dump(payload, _f)
    chunk_sizes.append(this_size)
    per_chunk_id_checksum[chunk_id] = _ids_checksum(chunk_idxs)
    print(f"  chunk{chunk_id:02d}  [{start:>7,} \u2013 {end-1:>7,}]  {this_size:>6,} entries")
    start = end

# ── Write manifest (counts + checksums, NOT the full structure-ID list -- the
#    full IDs already exist inside each chunk's 'structure_ids' field, so
#    duplicating them here would just be redundant weight) ───────────────────
manifest = {
    'functional':                      CHUNK_FUNCTIONAL,
    'total_in_dataset':                total_in_dataset,
    'total_used':                      total_used,
    'n_chunks':                        CHUNK_N,
    'chunk_sizes':                     chunk_sizes,
    'dataset_structure_id_checksum':   _ids_checksum(all_indices),
    'per_chunk_structure_id_checksum': per_chunk_id_checksum,
}
manifest_file = os.path.join(CHUNK_BASE_DIR, f'{functional_lower}_manifest.json')
with open(manifest_file, 'w') as _f:
    json.dump(manifest, _f, indent=2)

print(f"\nAll {CHUNK_N} chunks written to: {CHUNK_BASE_DIR}")
print(f"Manifest: {manifest_file}")

In [1]:
import os, stat, json, glob, textwrap

---
## Section 1 — Imports & Helpers

The helper functions defined here — `load_json_gz`/`save_json`, `safe_filename`, `angle_between`, `nan_to_none`, `get_static_energy`, `load_all_entries`, and the checksummed `compute_force_results_for_structure` calculation block — are the same code every generated `run_chunk_XX.py` runs on the cluster.  
They are stored as a string (`HELPERS_CODE`) so they can be embedded verbatim into every generated run script — ensuring the scripts and the interactive notebook share exactly the same code.

In [ ]:
# FORCE_RESULTS_CALC_CHECKSUM is defined twice on purpose: once here as a real
# notebook-level constant (used by the merge cells below to validate that every
# merged chunk was produced by the same calculation logic), and once again
# inside the HELPERS_CODE string (so every generated run_chunk_XX.py carries its
# own copy with no FPBench import required on the cluster). Keep both in sync.
FORCE_RESULTS_CALC_CHECKSUM = "18453991f7f0a4e33be95d14d33e626081ce0b88ca279c0a9135639369d04387"

HELPERS_CODE = textwrap.dedent("""
    import os, json, gzip, re, time
    import numpy as np
    from pymatgen.core import Structure
    from pymatgen.io.ase import AseAtomsAdaptor

    _adaptor = AseAtomsAdaptor()


    def load_json_gz(path):
        with gzip.open(path, 'rt') as f:
            return json.load(f)


    def save_json(obj, path):
        with open(path, 'w') as f:
            json.dump(obj, f, indent=2, allow_nan=False)


    def get_entries(raw):
        return raw.get('data', raw) if isinstance(raw, dict) else raw


    def safe_filename(s):
        import re as _re
        return _re.sub(r'[^\w\\-_.]', '_', s)


    def angle_between(v1, v2):
        n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
        if n1 < 1e-8 or n2 < 1e-8:
            return float('nan')
        return np.degrees(np.arccos(np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0)))


    def nan_to_none(x):
        # JSON has no NaN token; convert float('nan') -> None so allow_nan=False succeeds.
        if x is None:
            return None
        try:
            if np.isnan(x):
                return None
        except TypeError:
            pass
        return x


    def capture_runtime_versions():
        # Records what actually ran this chunk (python + the FP-relevant
        # packages available in *this* environment), independent of whatever
        # the registry *says* should be installed -- a cheap cross-check.
        import sys as _sys
        versions = {"python": _sys.version.split()[0], "python_executable": _sys.executable}
        for _pkg in ("numpy", "ase", "pymatgen", "torch", "matgl", "mace", "chgnet", "fairchem"):
            try:
                _mod = __import__(_pkg)
                versions[_pkg] = getattr(_mod, "__version__", "unknown")
            except ImportError:
                pass
        return versions


    def get_static_energy(structure, calc):
        from pymatgen.core import Structure as _Structure
        if isinstance(structure, _Structure):
            atoms = _adaptor.get_atoms(structure)
        else:
            atoms = structure          # already ASE Atoms
        atoms.calc = calc
        return {
            'energy': float(atoms.get_potential_energy()),
            'forces': atoms.get_forces().tolist(),
        }


    def _xyz_to_entries(xyz_path, n_max=None):
        from ase.io import read as _ase_read
        atoms_list = _ase_read(xyz_path, index=':')
        if n_max is not None:
            atoms_list = atoms_list[:n_max]
        entries = []
        for atoms in atoms_list:
            try:
                forces = atoms.get_forces().tolist()
                energy = float(atoms.get_potential_energy())
            except Exception:
                forces = [[0.0, 0.0, 0.0]] * len(atoms)
                energy = 0.0
            structure = _adaptor.get_structure(atoms)
            entries.append({
                'forces':         forces,
                'energy':         energy,
                'formula_pretty': atoms.get_chemical_formula(),
                'structure':      structure.as_dict(),
            })
        return entries


    def load_all_entries(data_source, n_chunks=20, n_max=None, functional_prefix=None):
        if os.path.isfile(data_source):
            if data_source.lower().endswith('.xyz'):
                return _xyz_to_entries(data_source, n_max=n_max)
            else:
                loader = load_json_gz if data_source.endswith('.gz') else (
                    lambda p: json.load(open(p))
                )
                all_entries = get_entries(loader(data_source))
        else:
            prefix = functional_prefix if functional_prefix is not None else FUNCTIONAL_PREFIX
            all_entries = []
            for idx in range(n_chunks):
                path = os.path.join(data_source, f'chunk{idx:02d}', f'{prefix}_{idx:02d}.json.gz')
                all_entries.extend(get_entries(load_json_gz(path)))
                if n_max is not None and len(all_entries) >= n_max:
                    break
        if n_max is not None:
            all_entries = all_entries[:n_max]
        return all_entries

    def compute_force_results_for_structure(F_dft, F_fp):
        # Standalone reimplementation of the per-structure core of build_force_results()
        # (see FPBench/force_results.py) -- identical formulas, identical zero-force
        # epsilon, kept in sync manually across generators (see FORCE_RESULTS_CALC_CHECKSUM).
        #
        # F_dft, F_fp : array-like, shape (n_atoms, 3) -- Cartesian forces, eV/A
        # returns a dict with keys: dft_force_magnitude, fp_force_magnitude,
        # force_magnitude_error, force_angle_error, force_vector_error -- each a
        # list of length n_atoms. force_angle_error entries are float('nan') where
        # the angle is undefined (near-zero force on either side); callers must
        # convert to None before writing JSON.
        F_dft  = np.asarray(F_dft,  dtype=float)
        F_fp = np.asarray(F_fp, dtype=float)
        if F_dft.shape != F_fp.shape:
            raise ValueError(
                f"DFT/FP force shape mismatch: {F_dft.shape} vs {F_fp.shape}"
            )
        if F_dft.ndim != 2 or F_dft.shape[1] != 3:
            raise ValueError(f"Expected (n_atoms, 3) Cartesian forces, got shape {F_dft.shape}")

        ZERO_FORCE_TOL = 1e-12

        mag_dft = np.linalg.norm(F_dft, axis=-1)
        mag_fp  = np.linalg.norm(F_fp, axis=-1)
        dF      = mag_fp - mag_dft
        e_vec   = np.linalg.norm(F_fp - F_dft, axis=-1)

        denom      = mag_dft * mag_fp
        safe_denom = np.where(denom > ZERO_FORCE_TOL, denom, 1.0)
        cosine     = np.einsum("ij,ij->i", F_dft, F_fp) / safe_denom
        cosine     = np.clip(cosine, -1.0, 1.0)
        theta      = np.degrees(np.arccos(cosine))
        theta      = np.where(denom > ZERO_FORCE_TOL, theta, np.nan)

        return {
            "dft_force_magnitude":   mag_dft.tolist(),
            "fp_force_magnitude":    mag_fp.tolist(),
            "force_magnitude_error": dF.tolist(),
            "force_angle_error":     theta.tolist(),
            "force_vector_error":    e_vec.tolist(),
        }


    # Kept in sync manually with FPBench/force_results.py's build_force_results().
    # If this ever mismatches FORCE_RESULTS_CALC_CHECKSUM below (defined identically
    # at notebook level, outside this string), the embedded calc logic has drifted.
    FORCE_RESULTS_CALC_CHECKSUM = "18453991f7f0a4e33be95d14d33e626081ce0b88ca279c0a9135639369d04387"
""").strip()

print("HELPERS_CODE defined.")

---
## Section 2 — Potential Registry

One shared registry entry per foundation potential, each carrying everything needed to run it as a standalone cluster job:
- `fp_name` — identifier used in output filenames
- `venv_activate` — path to the venv's `activate` script (sourced in SLURM header)
- `python` — full path to the venv Python binary
- `model_path` — path to the pre-trained model (`None` for built-in models like CHGNet)
- `calc_setup_code` — snippet that defines `calc` and is injected verbatim into the generated run script

**To add a new potential:** copy the commented template at the bottom, rename the key, and fill in your paths.

In [ ]:
from pathlib import Path

POTENTIAL_REGISTRY = {

    # ── CHGNet ────────────────────────────────────────────────────────────────
    "chgnet": {
        "fp_name":       "chgnet",
        "venv_activate":   Path("/path/to/chgnet_env/bin/activate"),
        "python":          Path("/path/to/chgnet_env/bin/python"),
        "model_path":      None,
        "calc_setup_code": textwrap.dedent("""
            from chgnet.model.dynamics import CHGNetCalculator
            calc = CHGNetCalculator()
        """).strip(),
    },

    # ── TensorNet (MatPES PBE v2025.1) ─────────────────────────────────────
    "tensornet_pbe": {
        "fp_name":       "TensorNET_matpes_PBE",
        "venv_activate":   Path("/path/to/m3gnet_env/bin/activate"),
        "python":          Path("/path/to/m3gnet_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/TensorNet-MatPES-PBE-v2025.1-PES"),
        "calc_setup_code": textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },
    
    # ── TensorNet (MatPES r2SCAN v2025.1) ─────────────────────────────────────
    "tensornet_r2scan": {
        "fp_name":       "TensorNET_matpes_R2SCAN",
        "venv_activate":   Path("/path/to/m3gnet_env/bin/activate"),
        "python":          Path("/path/to/m3gnet_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/TensorNet-MatPES-r2SCAN-v2025.1-PES"),
        "calc_setup_code": textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── MACE (MPA-0 medium) ────────────────────────────────────────────────────
    "mace": {
        "fp_name":       "Mace-MP0_medium",
        "venv_activate":   Path("/path/to/mace_env/bin/activate"),
        "python":          Path("/path/to/mace_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/mace-mpa-0-medium.model"),
        "calc_setup_code": textwrap.dedent("""
            from mace.calculators import MACECalculator
            calc = MACECalculator(
                model_paths=[MODEL_PATH],
                device="cpu",
                default_dtype="float64",
            )
        """).strip(),
    },

    # ── M3GNet (MatPES-PBE v2025.1) ───────────────────────────────────────────
    "m3gnet_matpes_pbe": {
        "fp_name":       "m3gnet_matpes_pbe",
        "venv_activate":   Path("/path/to/m3gnet_env/bin/activate"),
        "python":          Path("/path/to/m3gnet_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/M3GNet-MatPES-PBE-v2025.1-PES"),
        "calc_setup_code": textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },
    
    # ── M3GNet (MatPES-r2scan v2025.1) ───────────────────────────────────────────
    "m3gnet_matpes_r2scan": {
        "fp_name":       "m3gnet_matpes_r2scan",
        "venv_activate":   Path("/path/to/m3gnet_env/bin/activate"),
        "python":          Path("/path/to/m3gnet_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/M3GNet-MatPES-r2SCAN-v2025.1-PES"),
        "calc_setup_code": textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── M3GNet (MP-2021.2.8) ──────────────────────────────────────────────────
    "m3gnet_mp": {
        "fp_name":       "m3gnet_pes",
        "venv_activate":   Path("/path/to/m3gnet_env/bin/activate"),
        "python":          Path("/path/to/m3gnet_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/M3GNet-MP-2021.2.8-PES"),
        "calc_setup_code": textwrap.dedent("""
            from matgl.utils.io import load_model
            from matgl.ext.ase import PESCalculator
            calc = PESCalculator(potential=load_model(MODEL_PATH))
        """).strip(),
    },

    # ── UMA (FAIRChem / Meta) ─────────────────────────────────────────────────
    "uma": {
        "fp_name":       "UMA_s1_p1",
        "venv_activate":   Path("/path/to/uma_env/bin/activate"),
        "python":          Path("/path/to/uma_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/uma-s-1p1.pt"),
        "calc_setup_code": textwrap.dedent("""
            from fairchem.core import FAIRChemCalculator
            from fairchem.core.units.mlip_unit import load_predict_unit
            calc = FAIRChemCalculator(
                load_predict_unit(MODEL_PATH, device="cpu"),
                task_name="omat",
            )
        """).strip(),
    },
    
    # ── MACE-MatPES-PBE (omat fine-tuned) ─────────────────────────────────────
    "mace_matpes_pbe": {
        "fp_name":       "mace_matpes_pbe",
        "venv_activate":   Path("/path/to/mace_env/bin/activate"),
        "python":          Path("/path/to/mace_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/MACE-matpes-pbe-omat-ft.model"),
        "calc_setup_code": textwrap.dedent("""
            from mace.calculators import MACECalculator
            calc = MACECalculator(
                model_paths=[MODEL_PATH],
                device="cpu",
                default_dtype="float64",
            )
        """).strip(),
    },
    
    # ── MACE-MatPES-R2SCAN (omat fine-tuned) ─────────────────────────────────────
    "mace_matpes_r2scan": {
        "fp_name":       "mace_matpes_r2scan",
        "venv_activate":   Path("/path/to/mace_env/bin/activate"),
        "python":          Path("/path/to/mace_env/bin/python"),
        "model_path":      Path("/path/to/checkpoints/MACE-matpes-r2scan-omat-ft.model"),
        "calc_setup_code": textwrap.dedent("""
            from mace.calculators import MACECalculator
            calc = MACECalculator(
                model_paths=[MODEL_PATH],
                device="cpu",
                default_dtype="float64",
            )
        """).strip(),
    },

    # ── Custom / New Potential ────────────────────────────────────────────────
    # "my_new_fp": {
    #     "fp_name":       "my_new_fp",
    #    #     "venv_activate":   Path("/path/to/venv/bin/activate"),ar
    #     "python":          Path("/path/to/venv/bin/python"),
    #     "model_path":      Path("/path/to/model"),   # or None if built-in
    #     "calc_setup_code": "from mypackage import MyCalc\ncalc = MyCalc(MODEL_PATH)",
    # },
}

print("Available potentials:", list(POTENTIAL_REGISTRY.keys()))


# ── Dataset-specific potential selection (one shared registry above; each
#    generator activates only the subset relevant to its own dataset) ────────
MATPES_R2SCAN_POTENTIALS = [
    "m3gnet_matpes_r2scan",
    "tensornet_r2scan",
    "mace_matpes_r2scan",
]

print("MATPES_R2SCAN_POTENTIALS:", MATPES_R2SCAN_POTENTIALS)


---
## Section 3 — Configuration

Same as the PBE run generator, with two r2SCAN-specific settings:
- `FUNCTIONAL_PREFIX = "r2scan"` — matches the chunk filenames `r2scan_XX.json.gz` on HPC
- `DATASET_LABEL = "r2SCAN"` — used in output file names: `data_{fp}_r2SCAN_chunk_XX.json`
- `RUN_MODE` — `"chunked"` (one SLURM job per chunk) or `"single"` (one job for all chunks)
- `SLURM` — resource settings applied to every SLURM submit script

Each chunk file is expected at: `BASE_DATA_DIR/chunk{XX}/r2scan_{XX}.json.gz`

In [ ]:
from pathlib import Path

def _require_configured(value, name):
    s = str(value)
    if value is None or s.startswith("/path/to/") or s.startswith("your-"):
        raise ValueError(
            f"{name} has not been configured -- edit this cell and set {name} to "
            "the real value for your system before running this notebook."
        )
    return value

ACTIVE_POTENTIAL  = "tensornet_r2scan"    # key from POTENTIAL_REGISTRY -- must be an r2SCAN-trained model for this dataset
RUN_MODE          = "chunked"    # "chunked" or "single"
N_PARTS           = 20           # number of chunks
FUNCTIONAL_PREFIX = "r2scan"     # chunk filename prefix — matches r2scan_XX.json.gz on HPC
DATASET_LABEL     = "r2SCAN"     # label used in output filenames (data_*_r2SCAN_chunk_XX.json)
NOTEBOOK_NAME     = "matpes_r2scan_run_generator.ipynb"   # recorded in generation_metadata
STANDARDIZED_DATASET_NAME = "matpes_r2scan"    # -> {STANDARDIZED_DATASET_NAME}_force_results_standardized.json
STANDARDIZED_OUTPUT_DIR   = Path(".")          # where the final standardized JSON (Section 7) is written

# ── EDIT THESE FOR YOUR SYSTEM ────────────────────────────────────────────────
BASE_DATA_DIR     = Path("/path/to/dataset_chunks")     # must match Section 0's CHUNK_BASE_DIR
OUTPUT_DIR        = Path("/path/to/generated_results")  # where code_chunk_XX/ dirs and scripts are written
_require_configured(BASE_DATA_DIR, "BASE_DATA_DIR")
_require_configured(OUTPUT_DIR, "OUTPUT_DIR")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set DATA_SOURCE to override the default chunked layout.
# Options:
#   None                    -> use BASE_DATA_DIR with chunked layout (default)
#   "/path/to/file.xyz"     -> single XYZ file (all structures in one job)
#   "/path/to/data.json.gz" -> single JSON.gz file
DATA_SOURCE = None

SLURM = dict(
    account       = "your-slurm-account",   # EDIT: your cluster allocation/account
    ntasks        = 1,
    cpus_per_task = 4,
    partition     = "your-partition",       # EDIT: your cluster's partition/queue name
    mem_per_cpu   = "4G",
    time          = "144:00:00",
    email         = None,                   # EDIT: e.g. "you@example.edu" to enable job-completion emails, or leave None
    mail_type     = "END,FAIL",
)
_require_configured(SLURM["account"], "SLURM['account']")
_require_configured(SLURM["partition"], "SLURM['partition']")

cfg = POTENTIAL_REGISTRY[ACTIVE_POTENTIAL]
_data_source_display = DATA_SOURCE or BASE_DATA_DIR
print(f"Potential     : {cfg['fp_name']}")
print(f"Mode          : {RUN_MODE}")
print(f"N_PARTS       : {N_PARTS}")
print(f"Data source   : {_data_source_display}")
print(f"Func prefix   : {FUNCTIONAL_PREFIX}")
print(f"Dataset label : {DATASET_LABEL}")
print(f"Model path    : {cfg['model_path']}")
print(f"Output dir    : {OUTPUT_DIR}")

---
## Section 4 — Generate Scripts

Builds each run script by assembling three parts in order:
1. **Header** — `sys.path` insertion, config constants (`FP_NAME`, `BASE`, `MODEL_PATH`, `FUNCTIONAL_PREFIX`, `DATASET_LABEL`, chunk index)
2. **Calculator setup** — the `calc_setup_code` snippet from the registry
3. **Helpers + evaluation loop** — `HELPERS_CODE` from Section 1 + the evaluation loop

In `"chunked"` mode: creates `code_chunk_XX/run_chunk_XX.py` + `code_chunk_XX/submit_chunk_XX.slurm` + `mass_submit.sh`  
In `"single"` mode: creates `run_all_{fp}.py` + `submit_all_{fp}.slurm`

**Output files per chunk:** `data_{fp}_r2SCAN_chunk_XX.json` and `results_{fp}_r2SCAN_chunk_XX.json`

In [ ]:
# ── Evaluation loop — shared across all matpes_*_run_generator notebooks ─────
# CHUNK_IDX is a fixed int (chunked mode) or a loop variable (single mode).
# DATASET_LABEL (e.g. "r2SCAN") is injected via the script header.
EVAL_LOOP = textwrap.dedent("""
    if os.path.isfile(DATA_SOURCE):
        # Direct file (.xyz / .json / .json.gz) — load all entries in one shot
        entries = load_all_entries(DATA_SOURCE)
    else:
        # Chunked directory layout
        chunk_path = os.path.join(BASE, f"chunk{CHUNK_IDX:02d}", f"{FUNCTIONAL_PREFIX}_{CHUNK_IDX:02d}.json.gz")
        entries = get_entries(load_json_gz(chunk_path))
    print(f" Loaded {len(entries)} structures")

    energies_fp    = []
    forces_fp      = []
    original_indices = []   # kept for backward compatibility
    structure_ids    = []   # == original_indices; explicit name for downstream loaders
    results          = []

    # Metric-valid atoms only (force_angle_error is finite -- neither DFT nor FP
    # force is ~zero for that atom). This is the atom population every flattened
    # analysis array below is built from.
    all_structure_ids          = []
    all_atom_indices           = []
    all_dft_force_magnitude    = []
    all_fp_force_magnitude     = []
    all_force_magnitude_error  = []
    all_force_angle_error      = []
    all_force_vector_error     = []

    # Raw, UNFILTERED, DFT-only per-atom population (every atom, every structure).
    # Identical across every potential evaluated on this dataset -- used to build
    # reference_population in the Section 7 standardized merge, independent of
    # any one model's zero-FP-force exclusions.
    all_raw_structure_ids       = []
    all_raw_atom_indices        = []
    all_raw_dft_force_magnitude = []

    raw_atom_count = 0

    for i, data in enumerate(entries):
        forces_dft        = data["forces"]
        energy_dft        = data["energy"]
        structure_formula = safe_filename(data["formula_pretty"])
        structure         = Structure.from_dict(data["structure"])

        if "structure_id" in data:
            structure_id = data["structure_id"]
        elif "original_index" in data:
            structure_id = data["original_index"]
        else:
            raise KeyError(
                f"Entry {i} in this chunk has neither 'structure_id' nor "
                "'original_index' -- cannot determine structure identity. "
                "Refusing to silently fall back to the chunk-local loop index."
            )
        orig_idx = structure_id

        result                  = get_static_energy(structure, calc)
        fp_energy_structure_i = result["energy"]
        fp_forces_structure_i = result["forces"]

        energies_fp.append(fp_energy_structure_i)
        forces_fp.append(fp_forces_structure_i)
        original_indices.append(orig_idx)
        structure_ids.append(structure_id)

        calc_result = compute_force_results_for_structure(forces_dft, fp_forces_structure_i)
        dft_mag     = calc_result["dft_force_magnitude"]
        fp_mag      = calc_result["fp_force_magnitude"]
        dF          = calc_result["force_magnitude_error"]
        theta       = calc_result["force_angle_error"]   # float('nan') where undefined
        e_vec       = calc_result["force_vector_error"]

        n_atoms_struct  = len(dft_mag)
        raw_atom_count += n_atoms_struct

        # An atom's angle is undefined exactly when the DFT or FP force on it is
        # ~zero (see compute_force_results_for_structure) -- the manuscript
        # excludes those atoms from every flattened metric array, per-potential,
        # since force angle has no meaning there. The raw checkpoint (below,
        # `results`) still keeps every atom regardless.
        valid_metric_atom_indices = [a for a in range(n_atoms_struct) if np.isfinite(theta[a])]

        for a in valid_metric_atom_indices:
            all_structure_ids.append(structure_id)
            all_atom_indices.append(a)
            all_dft_force_magnitude.append(dft_mag[a])
            all_fp_force_magnitude.append(fp_mag[a])
            all_force_magnitude_error.append(dF[a])
            all_force_angle_error.append(theta[a])
            all_force_vector_error.append(e_vec[a])

        for a in range(n_atoms_struct):
            all_raw_structure_ids.append(structure_id)
            all_raw_atom_indices.append(a)
            all_raw_dft_force_magnitude.append(dft_mag[a])

        results.append({
            "structure_id":              structure_id,
            "original_index":            orig_idx,   # kept for backward compatibility
            "formula":                   structure_formula,
            "forces_dft":                np.array(forces_dft).tolist(),
            "fp_forces_structure_i":   np.array(fp_forces_structure_i).tolist(),
            "energy_dft":                float(energy_dft),
            "fp_energy_structure_i":   float(fp_energy_structure_i),
            "dft_force_magnitude":       dft_mag,   # full, unfiltered
            "fp_force_magnitude":        fp_mag,
            "force_magnitude_error":     dF,
            "force_angle_error":         [nan_to_none(t) for t in theta],   # null where undefined
            "force_vector_error":        e_vec,
            "valid_metric_atom_indices": valid_metric_atom_indices,
        })

    metric_valid_atom_count       = len(all_force_magnitude_error)
    excluded_undefined_angle_count = raw_atom_count - metric_valid_atom_count

    output = {
        "original_indices":    original_indices,   # kept for backward compatibility
        "structure_ids":       structure_ids,
        "energies_fp":       [float(e) for e in energies_fp],
        "forces_fp":         [np.array(f).tolist() for f in forces_fp],

        "all_structure_ids":         all_structure_ids,
        "all_atom_indices":          all_atom_indices,
        "all_dft_force_magnitude":   all_dft_force_magnitude,
        "all_fp_force_magnitude":    all_fp_force_magnitude,
        "all_force_magnitude_error": all_force_magnitude_error,
        "all_force_angle_error":     all_force_angle_error,
        "all_force_vector_error":    all_force_vector_error,

        "all_raw_structure_ids":         all_raw_structure_ids,
        "all_raw_atom_indices":          all_raw_atom_indices,
        "all_raw_dft_force_magnitude":   all_raw_dft_force_magnitude,

        "raw_atom_count":                 raw_atom_count,
        "metric_valid_atom_count":        metric_valid_atom_count,
        "excluded_undefined_angle_count": excluded_undefined_angle_count,
        "calc_block_checksum":            FORCE_RESULTS_CALC_CHECKSUM,
        "chunk_generated_at":             time.strftime("%Y-%m-%dT%H:%M:%S"),
        "runtime_environment":            capture_runtime_versions(),
    }

    tag = f"{FP_NAME}_{DATASET_LABEL}_chunk_{CHUNK_IDX:02d}"
    save_json(output,  f"data_{tag}.json")
    save_json(results, f"results_{tag}.json")
    print(f" Saved data_{tag}.json      ({metric_valid_atom_count:,} metric-valid atoms of {raw_atom_count:,} raw)")
    print(f" Saved results_{tag}.json   ({len(results):,} structures, all atoms retained)")
""").strip()


def make_run_script(fp_name, model_path, base, calc_setup,
                    data_source=None, chunk_idx=None, n_parts=None,
                    functional_prefix="r2scan", dataset_label="r2SCAN"):
    """
    Assemble a complete standalone Python run script from its parts.

    data_source    : str or None  -> if set, overrides chunked BASE path (e.g. .xyz file)
    chunk_idx      : int          -> chunked mode (processes one fixed chunk)
    n_parts        : int          -> single mode  (loops over range(n_parts))
    dataset_label  : str          -> label used in output filenames (e.g. 'r2SCAN')
    """
    resolved_data_source = data_source if data_source else base
    _ds_lower = resolved_data_source.lower()
    is_direct_file = _ds_lower.endswith('.xyz') or _ds_lower.endswith('.json') or _ds_lower.endswith('.json.gz')

    header = textwrap.dedent(f"""
        #!/usr/bin/env python3
        # Auto-generated by {NOTEBOOK_NAME}  |  Potential: {fp_name}
        #
        # Assumes the FP's own virtual environment is already active (see this
        # notebook's POTENTIAL_REGISTRY "venv_activate"/"python" for this
        # potential) -- no manual site-packages path is inserted here. The
        # correct FP environment provides its own packages.
        from __future__ import annotations

        FP_NAME          = "{fp_name}"
        BASE               = "{base}"
        DATA_SOURCE        = "{resolved_data_source}"
        MODEL_PATH         = "{model_path or ''}"
        FUNCTIONAL_PREFIX  = "{functional_prefix}"
        DATASET_LABEL      = "{dataset_label}"
    """).strip()

    if is_direct_file or chunk_idx is not None:
        effective_chunk_idx = chunk_idx if chunk_idx is not None else 0
        chunk_section = f"CHUNK_IDX = {effective_chunk_idx}"
        body          = EVAL_LOOP
        footer        = ""
    else:
        chunk_section = f"N_PARTS = {n_parts}"
        indented_loop = textwrap.indent(EVAL_LOOP, "    ")
        body          = f"for CHUNK_IDX in range(N_PARTS):\n    print(f\"\\n{'='*55}\\n Chunk {{CHUNK_IDX:02d}} / {n_parts-1:02d}\\n{'='*55}\")\n{indented_loop}"
        footer        = 'print("\\nAll chunks done.")'

    parts = [header, "", calc_setup, "", HELPERS_CODE, "", chunk_section, "", body]
    if footer:
        parts += ["", footer]

    return "\n".join(parts) + "\n"


# ── SLURM submit script template ──────────────────────────────────────────────
SLURM_TEMPLATE = textwrap.dedent("""
    #!/bin/bash
    #SBATCH --job-name={job_id}
    #SBATCH -A {account}
    #SBATCH --ntasks={ntasks}
    #SBATCH --cpus-per-task={cpus_per_task}
    #SBATCH -p {partition}
    #SBATCH --mem-per-cpu={mem_per_cpu}
    #SBATCH -t {time}
    #SBATCH --output=slurm_%j.out
    # Uncomment the two lines below (drop the leading "# " so they read
    # "#SBATCH ...") and set SLURM["email"] above to get job-completion emails:
    # SBATCH --mail-user={email}
    # SBATCH --mail-type={mail_type}

    source {venv_activate}
    {python} {script_name}
""").lstrip()

# ── Mass-submit script (chunked mode only) ────────────────────────────────────
MASS_SUBMIT = textwrap.dedent("""
    #!/bin/bash
    set -euo pipefail
    for d in code_chunk_*; do
      [ -d "$d" ] || continue
      idx="${d##code_chunk_}"
      echo "Submitting $d/submit_chunk_${idx}.slurm"
      (cd "$d" && sbatch "submit_chunk_${idx}.slurm")
    done
""").lstrip()

print("Script builder and templates defined.")

In [ ]:
def _write_executable(path, content):
    with open(path, "w") as f:
        f.write(content)
    os.chmod(path, os.stat(path).st_mode | stat.S_IXUSR)


cfg            = POTENTIAL_REGISTRY[ACTIVE_POTENTIAL]
fp_name      = cfg["fp_name"]
model_path_str = cfg["model_path"] or ""
data_source    = DATA_SOURCE  # None -> fall back to BASE_DATA_DIR in make_run_script

# All output for this potential goes under OUTPUT_DIR/{fp_name}/
potential_dir = os.path.join(OUTPUT_DIR, fp_name)
os.makedirs(potential_dir, exist_ok=True)

if RUN_MODE == "chunked":
    for i in range(N_PARTS):
        code_dir = os.path.join(potential_dir, f"code_chunk_{i:02d}")
        os.makedirs(code_dir, exist_ok=True)

        run_path = os.path.join(code_dir, f"run_chunk_{i:02d}.py")
        _write_executable(run_path, make_run_script(
            fp_name         = fp_name,
            model_path        = model_path_str,
            base              = BASE_DATA_DIR,
            calc_setup        = cfg["calc_setup_code"],
            data_source       = data_source,
            chunk_idx         = i,
            functional_prefix = FUNCTIONAL_PREFIX,
            dataset_label     = DATASET_LABEL,
        ))

        slurm_path = os.path.join(code_dir, f"submit_chunk_{i:02d}.slurm")
        _write_executable(slurm_path, SLURM_TEMPLATE.format(
            job_id        = f"{fp_name}_{i:02d}",
            venv_activate = cfg["venv_activate"],
            python        = cfg["python"],
            script_name   = os.path.basename(run_path),
            **SLURM,
        ))

    _write_executable(os.path.join(potential_dir, "mass_submit.sh"), MASS_SUBMIT)

    print(f"Generated {N_PARTS} chunk directories in: {potential_dir}/")
    print(f"  {fp_name}/code_chunk_00/ ... {fp_name}/code_chunk_{N_PARTS-1:02d}/")
    print(f"  {fp_name}/mass_submit.sh")
    print(f"\nNext: rsync to cluster, then:  cd {fp_name} && bash mass_submit.sh")

elif RUN_MODE == "single":
    script_name = f"run_all_{fp_name}.py"
    run_path    = os.path.join(potential_dir, script_name)
    _write_executable(run_path, make_run_script(
        fp_name         = fp_name,
        model_path        = model_path_str,
        base              = BASE_DATA_DIR,
        calc_setup        = cfg["calc_setup_code"],
        data_source       = data_source,
        n_parts           = N_PARTS,
        functional_prefix = FUNCTIONAL_PREFIX,
        dataset_label     = DATASET_LABEL,
    ))

    slurm_path = os.path.join(potential_dir, f"submit_all_{fp_name}.slurm")
    _write_executable(slurm_path, SLURM_TEMPLATE.format(
        job_id        = fp_name,
        venv_activate = cfg["venv_activate"],
        python        = cfg["python"],
        script_name   = script_name,
        **SLURM,
    ))

    print(f"Generated: {run_path}")
    print(f"           {slurm_path}")
    print(f"\nNext: rsync to cluster, then:  sbatch {os.path.basename(slurm_path)}")

else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE!r}  (must be 'chunked' or 'single')")

---
## Section 5 — Validate & Merge (One Potential)

Strict by default: refuses to merge if any expected chunk is missing, if a
chunk's calc-block checksum doesn't match, or if the merged structure_ids
contain a duplicate. Set `STRICT_MERGE = False` to merge whatever chunks
exist yet (e.g. to check progress mid-run). Writes
`all_results_{fp}_{DATASET_LABEL}.json`.

In `"chunked"` mode the output files land inside `code_chunk_XX/` subdirectories, so this cell searches recursively.

In [ ]:
import hashlib, time

STRICT_MERGE = True   # False = merge whatever chunks exist yet (useful mid-run on the cluster)

def _ids_checksum(ids):
    return hashlib.sha256(",".join(str(x) for x in sorted(ids)).encode("utf-8")).hexdigest()

def _raw_dft_checksum(structure_ids, atom_indices, dft_mags):
    # Order-independent checksum over (structure_id, atom_index, dft_magnitude)
    # triples -- identical DFT reference data should produce an identical
    # checksum regardless of which FP model (or chunk order) produced it.
    rows = sorted(zip(structure_ids, atom_indices, dft_mags), key=lambda t: (str(t[0]), t[1]))
    payload = "|".join(f"{s}:{a}:{d:.10g}" for s, a, d in rows)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

_MERGE_FIELDS = (
    "original_indices", "structure_ids", "energies_fp", "forces_fp",
    "all_structure_ids", "all_atom_indices",
    "all_dft_force_magnitude", "all_fp_force_magnitude",
    "all_force_magnitude_error", "all_force_angle_error", "all_force_vector_error",
    "all_raw_structure_ids", "all_raw_atom_indices", "all_raw_dft_force_magnitude",
)

def merge_one_potential(registry_key, output_dir, dataset_label, n_parts, strict=True,
                          notebook_name=None):
    """
    Validate and merge one potential's per-chunk data_*.json files.

    Returns the merged dict (with a "generation_metadata" key added), or None
    if no chunk files exist yet. Raises RuntimeError on any consistency
    violation (missing/duplicate chunks under strict=True, duplicate
    structure_ids, calc-block checksum mismatch, or field-length mismatches).

    Does NOT require any particular *count* of metric-valid atoms -- a
    potential's own zero-FP-force exclusions are expected to differ from
    other potentials' and are not an error.
    """
    fp_name = POTENTIAL_REGISTRY[registry_key]["fp_name"]

    def _find(kind, idx):
        # Restricted to OUTPUT_DIR/<fp_name>/ -- never the whole OUTPUT_DIR tree --
        # so this can never accidentally pick up another potential's (or an old/
        # stale) results directory that happens to contain a similarly-named file.
        return glob.glob(os.path.join(
            output_dir, fp_name, "**", f"{kind}_{fp_name}_{dataset_label}_chunk_{idx:02d}.json"
        ), recursive=True)

    missing, dupes, data_files = [], [], {}
    for idx in range(n_parts):
        d = _find("data", idx)
        if not d:
            missing.append(idx)
            continue
        if len(d) > 1:
            dupes.append(idx)
            continue
        data_files[idx] = d[0]

    if dupes:
        raise RuntimeError(f"[{fp_name}] duplicate chunk output files for chunk index(es): {dupes}")
    if missing and strict:
        raise RuntimeError(
            f"[{fp_name}] missing data_*.json for chunk index(es) {missing} out of "
            f"{n_parts} expected. Set strict=False to merge the available chunks anyway."
        )
    if not data_files:
        return None

    merged = {k: [] for k in _MERGE_FIELDS}
    checksums_seen  = set()
    raw_atom_count           = 0
    metric_valid_atom_count  = 0
    chunk_generated_at       = []
    runtime_environment_sample = None

    for idx in sorted(data_files):
        with open(data_files[idx]) as f:
            chunk = json.load(f)
        checksums_seen.add(chunk.get("calc_block_checksum"))
        for key in _MERGE_FIELDS:
            merged[key].extend(chunk[key])
        raw_atom_count          += chunk["raw_atom_count"]
        metric_valid_atom_count += chunk["metric_valid_atom_count"]
        if chunk.get("chunk_generated_at"):
            chunk_generated_at.append(chunk["chunk_generated_at"])
        if runtime_environment_sample is None and chunk.get("runtime_environment"):
            runtime_environment_sample = chunk["runtime_environment"]

    if checksums_seen - {FORCE_RESULTS_CALC_CHECKSUM}:
        raise RuntimeError(
            f"[{fp_name}] calc-block checksum mismatch across merged chunks "
            f"(expected {FORCE_RESULTS_CALC_CHECKSUM!r}, found {checksums_seen!r}); "
            "refusing to merge results computed with different formulas."
        )

    seen_ids, dup_ids = set(), set()
    for sid in merged["structure_ids"]:
        if sid in seen_ids:
            dup_ids.add(sid)
        seen_ids.add(sid)
    if dup_ids:
        raise RuntimeError(f"[{fp_name}] duplicate structure_id(s) across merged chunks: {sorted(map(str, dup_ids))[:20]} ...")

    n_struct = len(merged["energies_fp"])
    for key in ("forces_fp", "original_indices", "structure_ids"):
        if len(merged[key]) != n_struct:
            raise RuntimeError(f"[{fp_name}] structure-level length mismatch: {key} has {len(merged[key])}, expected {n_struct}")

    n_metric = len(merged["all_force_magnitude_error"])
    for key in ("all_structure_ids", "all_atom_indices", "all_dft_force_magnitude",
                "all_fp_force_magnitude", "all_force_angle_error", "all_force_vector_error"):
        if len(merged[key]) != n_metric:
            raise RuntimeError(f"[{fp_name}] metric-valid atom-level length mismatch: {key} has {len(merged[key])}, expected {n_metric}")

    n_raw = len(merged["all_raw_dft_force_magnitude"])
    for key in ("all_raw_structure_ids", "all_raw_atom_indices"):
        if len(merged[key]) != n_raw:
            raise RuntimeError(f"[{fp_name}] raw atom-level length mismatch: {key} has {len(merged[key])}, expected {n_raw}")

    cfg = POTENTIAL_REGISTRY[registry_key]
    merged["generation_metadata"] = {
        "fp_name":                    fp_name,
        "registry_key":                 registry_key,
        "model_path":                   cfg["model_path"],
        "configured_python":            cfg["python"],
        "configured_venv_activate":     cfg["venv_activate"],
        "notebook_name":                notebook_name,
        "dataset_label":                dataset_label,
        "n_chunks_expected":            n_parts,
        "n_chunks_found":               len(data_files),
        "missing_chunk_indices":        missing,
        "partial":                      bool(missing),
        "raw_structure_count":          n_struct,
        "raw_atom_count":               raw_atom_count,
        "metric_valid_atom_count":      metric_valid_atom_count,
        "excluded_undefined_angle_count": raw_atom_count - metric_valid_atom_count,
        "structure_id_checksum":        _ids_checksum(merged["structure_ids"]),
        "raw_dft_checksum":             _raw_dft_checksum(merged["all_raw_structure_ids"], merged["all_raw_atom_indices"], merged["all_raw_dft_force_magnitude"]),
        "calc_block_checksum":          FORCE_RESULTS_CALC_CHECKSUM,
        "chunk_generated_at_range":     [min(chunk_generated_at), max(chunk_generated_at)] if chunk_generated_at else None,
        "runtime_environment_sample":   runtime_environment_sample,
        "merged_at":                    time.strftime("%Y-%m-%dT%H:%M:%S"),
    }
    return merged


merged = merge_one_potential(ACTIVE_POTENTIAL, OUTPUT_DIR, DATASET_LABEL, N_PARTS,
                              strict=STRICT_MERGE, notebook_name=NOTEBOOK_NAME)

if merged is None:
    print("No chunk files found yet — run the SLURM jobs first.")
else:
    fp_name = POTENTIAL_REGISTRY[ACTIVE_POTENTIAL]["fp_name"]
    out_path  = os.path.join(OUTPUT_DIR, f"all_results_{fp_name}_{DATASET_LABEL}.json")
    with open(out_path, "w") as f:
        json.dump(merged, f, indent=2, allow_nan=False)

    gm = merged["generation_metadata"]
    print(f"Merged {gm['n_chunks_found']}/{gm['n_chunks_expected']} chunk(s)" + ("  (PARTIAL)" if gm["partial"] else ""))
    print(f"  Raw structures             : {gm['raw_structure_count']:,}")
    print(f"  Raw atoms                  : {gm['raw_atom_count']:,}")
    print(f"  Metric-valid atoms         : {gm['metric_valid_atom_count']:,}")
    print(f"  Excluded (undefined angle) : {gm['excluded_undefined_angle_count']:,}")
    print(f"\nSaved: {out_path}")

---
## Section 6 — Generate + Submit (All Potentials)

Loops over every registry key in `MATPES_R2SCAN_POTENTIALS` (not the full shared `POTENTIAL_REGISTRY`, which also carries PBE/OMat24 entries used by the other generators) and writes each potential's scripts under `OUTPUT_DIR/{fp_name}/code_chunk_XX/`.

Also writes `submit_all_models.sh` (chunked mode only — see the guard in the cell below) and `registry_key_map.tsv`. When run on the cluster, `submit_all_models.sh` submits every chunk of every potential via `sbatch --parsable` and appends one row per job to `submitted_jobs.tsv` (`timestamp`, `registry_key`, `fp_name`, `chunk_id`, `SLURM_job_id`).

In [ ]:
# ── Generate scripts for EVERY potential in MATPES_R2SCAN_POTENTIALS ──────────────────
# Uses N_PARTS, FUNCTIONAL_PREFIX, DATASET_LABEL, BASE_DATA_DIR, RUN_MODE, and
# SLURM from Section 3. Each FP gets its own subdirectory under OUTPUT_DIR:
#   OUTPUT_DIR/{standardized_model_key}/code_chunk_XX/
# Only iterates over this dataset's own potentials list -- NOT the full shared
# POTENTIAL_REGISTRY, which also carries entries meant for the other datasets.

_potentials_to_skip = set()   # add registry keys here to skip specific potentials

generated = []
_registry_key_of = {}   # fp_name -> registry_key, for submitted_jobs.tsv below

for _reg_key in MATPES_R2SCAN_POTENTIALS:
    if _reg_key in _potentials_to_skip:
        print(f"  Skipping {_reg_key}")
        continue
    _pot_cfg = POTENTIAL_REGISTRY[_reg_key]
    _fp    = _pot_cfg["fp_name"]
    _registry_key_of[_fp] = _reg_key

    _fp_out = os.path.join(OUTPUT_DIR, _fp)
    os.makedirs(_fp_out, exist_ok=True)

    if RUN_MODE == "chunked":
        for i in range(N_PARTS):
            _code_dir = os.path.join(_fp_out, f"code_chunk_{i:02d}")
            os.makedirs(_code_dir, exist_ok=True)

            _run_path = os.path.join(_code_dir, f"run_chunk_{i:02d}.py")
            _write_executable(_run_path, make_run_script(
                fp_name         = _fp,
                model_path        = _pot_cfg["model_path"] or "",
                base              = BASE_DATA_DIR,
                calc_setup        = _pot_cfg["calc_setup_code"],
                data_source       = DATA_SOURCE,
                chunk_idx         = i,
                functional_prefix = FUNCTIONAL_PREFIX,
                dataset_label     = DATASET_LABEL,
            ))

            _slurm_path = os.path.join(_code_dir, f"submit_chunk_{i:02d}.slurm")
            _write_executable(_slurm_path, SLURM_TEMPLATE.format(
                job_id        = f"{_fp}_{i:02d}",
                venv_activate = _pot_cfg["venv_activate"],
                python        = _pot_cfg["python"],
                script_name   = os.path.basename(_run_path),
                **SLURM,
            ))

        _write_executable(os.path.join(_fp_out, "mass_submit.sh"), MASS_SUBMIT)

    elif RUN_MODE == "single":
        _script_name = f"run_all_{_fp}.py"
        _run_path    = os.path.join(_fp_out, _script_name)
        _write_executable(_run_path, make_run_script(
            fp_name         = _fp,
            model_path        = _pot_cfg["model_path"] or "",
            base              = BASE_DATA_DIR,
            calc_setup        = _pot_cfg["calc_setup_code"],
            data_source       = DATA_SOURCE,
            n_parts           = N_PARTS,
            functional_prefix = FUNCTIONAL_PREFIX,
            dataset_label     = DATASET_LABEL,
        ))
        _slurm_path = os.path.join(_fp_out, f"submit_all_{_fp}.slurm")
        _write_executable(_slurm_path, SLURM_TEMPLATE.format(
            job_id        = _fp,
            venv_activate = _pot_cfg["venv_activate"],
            python        = _pot_cfg["python"],
            script_name   = _script_name,
            **SLURM,
        ))

    generated.append(_fp)
    print(f"  {_fp:30s}  →  {_fp_out}")

# registry_key_map.tsv lets submit_all_models.sh (pure bash, no Python) look
# up each directory's registry key for the submitted_jobs.tsv log below.
with open(os.path.join(OUTPUT_DIR, "registry_key_map.tsv"), "w") as _f:
    _f.write("fp_name\tregistry_key\n")
    for _fp, _reg_key in _registry_key_of.items():
        _f.write(f"{_fp}\t{_reg_key}\n")

if RUN_MODE != "chunked":
    print(f"\nRUN_MODE={RUN_MODE!r}: submit_all_models.sh is only generated for "
          "RUN_MODE=\'chunked\' (it walks code_chunk_XX/ subdirectories, which "
          "single mode does not create). Submit each potential's "
          "submit_all_{fp}.slurm individually.")
else:
    _submit_all_path = os.path.join(OUTPUT_DIR, "submit_all_models.sh")
    _write_executable(_submit_all_path, textwrap.dedent('''
        #!/bin/bash
        # Auto-generated -- submits every chunk of every potential generated
        # above via `sbatch --parsable`, and appends one row per job to
        # submitted_jobs.tsv (timestamp, registry_key, fp_name, chunk_id, job_id).
        set -euo pipefail
        cd "$(dirname "$0")"
        log="submitted_jobs.tsv"
        [ -f "$log" ] || printf "timestamp\tregistry_key\tfp_name\tchunk_id\tSLURM_job_id\n" > "$log"

        lookup_registry_key() {
          awk -F"\t" -v m="$1" \'$1==m{print $2}\' registry_key_map.tsv
        }

        for d in */ ; do
          d="${d%/}"
          [ -f "$d/mass_submit.sh" ] || continue
          reg_key="$(lookup_registry_key "$d")"
          echo "Submitting all chunks for: $d  (registry_key=$reg_key)"
          for chunk_dir in "$d"/code_chunk_*/ ; do
            chunk_dir="${chunk_dir%/}"
            chunk_id="${chunk_dir##*_}"
            slurm_file="$chunk_dir/submit_chunk_${chunk_id}.slurm"
            [ -f "$slurm_file" ] || continue
            job_id=$(cd "$chunk_dir" && sbatch --parsable "$(basename "$slurm_file")")
            printf "%s\t%s\t%s\t%s\t%s\n" "$(date -Iseconds)" "$reg_key" "$d" "$chunk_id" "$job_id" >> "$log"
          done
        done
    ''').lstrip())

print(f"\nGenerated scripts for {len(generated)} potential(s): {generated}")
print(f"\nCluster workflow:")
print(f"  1. rsync -av {OUTPUT_DIR}/ cluster:{OUTPUT_DIR}/")
if RUN_MODE == "chunked":
    print(f"  2. On cluster: cd {OUTPUT_DIR} && bash submit_all_models.sh")
else:
    print(f"  2. On cluster: for each potential, sbatch {{output}}/submit_all_{{fp}}.slurm")

---
## Section 7 — Strict All-Model Merge + Standardized Output

Re-uses Section 5's `merge_one_potential()` for every potential in `MATPES_R2SCAN_POTENTIALS`, then cross-checks that every model saw the *same* raw DFT reference data (same structures, same order, same atom counts, same DFT force checksum) before combining them. Per-model zero-FP-force exclusions are expected to differ and are **not** treated as an inconsistency — only the raw, unfiltered population has to match across models.

Writes exactly one final file: `{STANDARDIZED_DATASET_NAME}_force_results_standardized.json` under `STANDARDIZED_OUTPUT_DIR`, containing `schema_version`, `dataset_name`, `units`, `generation_metadata`, a `reference_population` (built once from the raw DFT chunks, independent of any one FP model), and `models` (each potential's own metric-valid, per-atom arrays plus `structure_id`/`atom_index` provenance). The analysis notebooks keep loading `standardized["models"]` unmodified.

In [ ]:
# ── Strict all-model merge + standardized dataset output ─────────────────────
# Re-uses merge_one_potential() from Section 5 for every potential in
# MATPES_R2SCAN_POTENTIALS, then cross-checks that every model saw the *same* raw
# DFT reference data (same structures, same order, same atom counts, same DFT
# force values) before combining them into one standardized dataset file.
# Per-model zero-FP-force exclusions are expected to differ and are NOT
# treated as an inconsistency.

STRICT_MERGE_ALL = True   # False = skip incomplete/failed potentials instead of aborting

_per_model_merged = {}
_skipped = []

for _reg_key in MATPES_R2SCAN_POTENTIALS:
    _fp = POTENTIAL_REGISTRY[_reg_key]["fp_name"]
    try:
        _m = merge_one_potential(_reg_key, OUTPUT_DIR, DATASET_LABEL, N_PARTS,
                                  strict=STRICT_MERGE_ALL, notebook_name=NOTEBOOK_NAME)
    except RuntimeError as e:
        if STRICT_MERGE_ALL:
            raise RuntimeError(f"Section 7 aborted while merging {_fp}: {e}")
        print(f"  {_fp:30s}  — merge failed, skipping ({e})")
        _skipped.append(_fp)
        continue

    if _m is None:
        print(f"  {_fp:30s}  — no chunk files found yet, skipping")
        _skipped.append(_fp)
        continue

    _per_model_merged[_reg_key] = _m
    _gm = _m["generation_metadata"]
    print(f"  {_fp:30s}  {_gm['n_chunks_found']:2d}/{_gm['n_chunks_expected']} chunks  |  "
          f"{_gm['raw_structure_count']:,} structures  |  "
          f"{_gm['metric_valid_atom_count']:,}/{_gm['raw_atom_count']:,} metric-valid atoms")

if not _per_model_merged:
    raise RuntimeError("Section 7: no potentials merged successfully -- nothing to standardize.")

# ── Cross-model RAW consistency (must match -- same dataset, same chunks) ────
_ref_key    = next(iter(_per_model_merged))
_ref_merged = _per_model_merged[_ref_key]
_ref_gm     = _ref_merged["generation_metadata"]

for _reg_key, _m in _per_model_merged.items():
    _gm = _m["generation_metadata"]
    if _m["structure_ids"] != _ref_merged["structure_ids"]:
        raise RuntimeError(
            f"Structure sequence mismatch: {_ref_key} and {_reg_key} did not process the same "
            "ordered source-structure sequence (different chunk contents, order, or count)."
        )
    if _gm["raw_atom_count"] != _ref_gm["raw_atom_count"]:
        raise RuntimeError(f"Raw atom count mismatch: {_ref_key}={_ref_gm['raw_atom_count']} vs {_reg_key}={_gm['raw_atom_count']}")
    if _gm["raw_dft_checksum"] != _ref_gm["raw_dft_checksum"]:
        raise RuntimeError(
            f"Raw DFT reference checksum mismatch: {_ref_key} vs {_reg_key} -- DFT forces "
            "differ between these potentials' chunks, which should be impossible since DFT "
            "forces do not depend on the FP model. Check that both processed the same source data."
        )
    # NOTE: metric_valid_atom_count is intentionally NOT compared here --
    # different potentials legitimately exclude different zero-FP-force atoms.

# ── reference_population: built once from any one (now checksum-verified-
#    identical) model's RAW, unfiltered DFT-only arrays. ─────────────────────
reference_population = {
    "dft_force_magnitude": list(_ref_merged["all_raw_dft_force_magnitude"]),
    "structure_id":        list(_ref_merged["all_raw_structure_ids"]),
    "atom_index":          list(_ref_merged["all_raw_atom_indices"]),
}

models = {}
per_model_summary = {}
for _reg_key, _m in _per_model_merged.items():
    _fp = POTENTIAL_REGISTRY[_reg_key]["fp_name"]
    _gm   = _m["generation_metadata"]
    models[_fp] = {
        "dft_force_magnitude":   list(_m["all_dft_force_magnitude"]),
        "fp_force_magnitude":    list(_m["all_fp_force_magnitude"]),
        "force_magnitude_error": list(_m["all_force_magnitude_error"]),
        "force_angle_error":     list(_m["all_force_angle_error"]),
        "force_vector_error":    list(_m["all_force_vector_error"]),
        "structure_id":          list(_m["all_structure_ids"]),
        "atom_index":            list(_m["all_atom_indices"]),
    }
    per_model_summary[_fp] = {
        "registry_key":                    _reg_key,
        "n_chunks_found":                  _gm["n_chunks_found"],
        "n_chunks_expected":               _gm["n_chunks_expected"],
        "partial":                         _gm["partial"],
        "raw_atom_count":                  _gm["raw_atom_count"],
        "metric_valid_atom_count":         _gm["metric_valid_atom_count"],
        "excluded_undefined_angle_count":  _gm["excluded_undefined_angle_count"],
        "model_path":                      _gm["model_path"],
        "configured_python":               _gm["configured_python"],
        "configured_venv_activate":        _gm["configured_venv_activate"],
        "structure_id_checksum":           _gm["structure_id_checksum"],
        "chunk_generated_at_range":        _gm["chunk_generated_at_range"],
        "runtime_environment_sample":      _gm["runtime_environment_sample"],
        "merged_at":                       _gm["merged_at"],
    }

standardized = {
    "schema_version": "1.0",
    "dataset_name": STANDARDIZED_DATASET_NAME,
    "units": {"force": "eV/Å", "angle": "degree"},
    "generation_metadata": {
        "dataset_label":        DATASET_LABEL,
        "functional_prefix":    FUNCTIONAL_PREFIX,
        "notebook_name":        NOTEBOOK_NAME,
        "n_chunks_expected":    N_PARTS,
        "models_included":      list(models.keys()),
        "models_skipped":       _skipped,
        "raw_structure_count":  _ref_gm["raw_structure_count"],
        "raw_atom_count":       _ref_gm["raw_atom_count"],
        "raw_dft_checksum":     _ref_gm["raw_dft_checksum"],
        "calc_block_checksum":  FORCE_RESULTS_CALC_CHECKSUM,
        "merged_at":            time.strftime("%Y-%m-%dT%H:%M:%S"),
        "per_model_summary":    per_model_summary,
    },
    "reference_population": reference_population,
    "models": models,
}

os.makedirs(STANDARDIZED_OUTPUT_DIR, exist_ok=True)
_out_path = os.path.join(STANDARDIZED_OUTPUT_DIR, f"{STANDARDIZED_DATASET_NAME}_force_results_standardized.json")
with open(_out_path, "w") as f:
    json.dump(standardized, f, indent=2, allow_nan=False)

print(f"\n{chr(0x2500)*70}")
print(f"Standardized dataset file written: {_out_path}")
print(f"  Models included: {list(models.keys())}")
if _skipped:
    print(f"  Models skipped : {_skipped}")